In [1]:
import os
import sys
import numpy as np
import torch
from dotenv import load_dotenv
from torch.utils.data import DataLoader

In [2]:
pwd

'/home/afan2025/Masters/Deep_Learning_Systems/CSMC_35200_BraintoTextGroup/DiffNeuralDecoder/diffusionNeuralDecoder/scripts'

In [3]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('/home/afan2025/Masters/Deep_Learning_Systems/CSMC_35200_BraintoTextGroup/DiffNeuralDecoder/diffusionNeuralDecoder/scripts/check_embeddings_notebook.ipynb'))        # .../diffusionNeuralDecoder/scripts
PROJECT_DIR = os.path.dirname(SCRIPT_DIR)                      # .../diffusionNeuralDecoder
REPO_DIR = os.path.dirname(PROJECT_DIR)                        # .../DiffNeuralDecoder
LOG_DIR = os.path.join(PROJECT_DIR, "logs")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

In [4]:
load_dotenv(os.path.join(PROJECT_DIR, ".env"))

True

In [6]:
from diffusion_model import PhonemeDiT
from diffusion import create_diffusion
from diffusionNeuralDecoder.datasets.speechDataset import BrainToTextDataset, ID_TO_PHONE
from scripts.pretrain import (
    _get_env,
    _resolve_path,
)
from scripts.brain_finetune import _batch_loss

In [7]:
BASE_DIR = _get_env("BASE_DIR", default=PROJECT_DIR)
COMPETITION_DATA_DIR = _resolve_path(BASE_DIR, _get_env("COMPETITION_DATA_DIR", default="../../../competition_data"))
PREPROCESSED_DATA_DIR = _get_env("PREPROCESSED_DATA_DIR", default="/net/scratch/afan2025/preprocessed_data")
CHECKPOINT_DIR = _resolve_path(BASE_DIR, _get_env("CHECKPOINT_DIR", default="./checkpoints"))

Z_BRAIN_DIM = _get_env("Z_BRAIN_DIM", int)
D_MODEL = _get_env("D_MODEL", int)
MAX_TEXT_LEN = _get_env("MAX_TEXT_LEN", int)
VOCAB_SIZE = _get_env("VOCAB_SIZE", int)
MODEL_DEPTH = _get_env("MODEL_DEPTH", int)
NUM_HEADS = _get_env("NUM_HEADS", int)
MLP_RATIO = _get_env("MLP_RATIO", float)
DECODER_METHOD = _get_env("DECODER_METHOD", default="nn")
DIFFUSION_NOISE_SCHEDULE = _get_env("DIFFUSION_NOISE_SCHEDULE", default="cosine")

In [10]:
torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [14]:
dataset = BrainToTextDataset(data_path=PREPROCESSED_DATA_DIR, partition = "train")
vocab_size = dataset.vocab_size
vocab_size

75

In [40]:
model = PhonemeDiT(
        d_model=D_MODEL,
        vocab_size=vocab_size,
        depth=MODEL_DEPTH,
        max_len=MAX_TEXT_LEN,
        num_heads=NUM_HEADS,
        mlp_ratio=MLP_RATIO,
        use_cross_attention= False,  
        z_brain_dim=Z_BRAIN_DIM,
        use_final_layer=False,
        ).to(device)

In [41]:
model.x_embedder.weight

Parameter containing:
tensor([[-0.0497, -0.0061, -0.0256,  ..., -0.0352, -0.0235, -0.0369],
        [-0.0315, -0.0063, -0.0126,  ..., -0.0064,  0.0062, -0.0165],
        [-0.0235, -0.0328, -0.0116,  ..., -0.0245, -0.0139, -0.0112],
        ...,
        [ 0.0148,  0.0193,  0.0170,  ...,  0.0060,  0.0194,  0.0174],
        [-0.0259, -0.0042,  0.0136,  ...,  0.0090,  0.0180,  0.0185],
        [ 0.0030,  0.0117, -0.0303,  ...,  0.0225,  0.0112, -0.0141]],
       requires_grad=True)

In [42]:
checkpoint_name = "best.pt"

ckpt_path = _resolve_path(BASE_DIR, os.path.join(CHECKPOINT_DIR, checkpoint_name))

In [43]:
model_checkpoint = torch.load(ckpt_path, map_location="cpu")
missing, unexpected = model.load_state_dict(model_checkpoint["model"], strict=False)

/tmp/ipykernel_2428974/3304516544.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_checkpoint = torch.load(ckpt_path, map_location="cpu")


In [44]:
model_checkpoint['model']

OrderedDict([('x_embedder.weight',
              tensor([[ 5.8745e-02,  7.6023e-03,  2.4728e-02,  ..., -2.1775e-02,
                       -3.3902e-02, -8.9698e-03],
                      [ 5.4581e-05,  1.1859e-05, -5.4538e-04,  ..., -3.7950e-04,
                       -3.2997e-04,  3.9470e-04],
                      [-3.1221e-04, -9.1055e-05, -3.0725e-04,  ..., -3.0772e-04,
                       -2.7472e-04,  2.1913e-04],
                      ...,
                      [-2.7988e-04, -5.4484e-05, -4.1720e-04,  ..., -1.8369e-04,
                       -3.2661e-04,  3.1900e-04],
                      [-2.5467e-04, -1.3379e-04, -3.5598e-04,  ..., -2.9524e-04,
                       -3.4894e-04,  2.2607e-04],
                      [-8.7337e-05, -5.4168e-05, -3.0534e-04,  ..., -2.6971e-04,
                       -3.0134e-04,  3.2718e-04]])),
             ('t_embedder.mlp.0.weight',
              tensor([[ 0.0295,  0.0185, -0.0022,  ...,  0.0064,  0.0109, -0.0175],
                      [-

In [48]:
E = model.x_embedder
E = E.weight.detach().numpy()

In [50]:
E.shape

(75, 256)

In [52]:
U, S, V = np.linalg.svd(E, full_matrices=False)
p_raw = S / (S.sum() + 1e-12)
p_raw_nz = p_raw[p_raw > 1e-12]
effective_rank_raw = float(np.exp(-np.sum(p_raw_nz * np.log(p_raw_nz))))

In [53]:
effective_rank_raw

5.216874122619629